In [79]:
raw_data = [
    "anaa lopezz desaparecida valencia",
    "AYUDA Luis Martinez CI 23456789 desaparecido en Caracas",
    "Se necesita rescate urgente familia atrapada en Valencia",
    "Centro de acopio en Barquisimeto recibiendo medicinas",
    "Maria Fernanda Gomez CI 87654321 desaparecida en Maracay",
    "Rescate urgente para niño en Mérida",
    "Centro de acopio en Chacao agua comida ropa",
    "Pedro Rodriguez CI 11223344 desaparecido en Valencia",
    "Se busca Ana Maria Perez CI 99887766 en Caracas",
    "Ayuda urgente rescate en Maracaibo",
    "Centro de acopio en Valencia alimentos",
    "Carlos Diaz CI 44556677 desaparecido en Maracay",
    "Rescate urgente en Barinas",
    "Centro de acopio en Caracas",
    "Luisana Rodriguez CI 55443322 desaparecida en Valencia",
    "Familia atrapada rescate urgente en Mérida",
    "Centro de acopio en Maracaibo",
    "Jose Perez CI 66778899 desaparecido en Caracas",
    "Ayuda rescate urgente en Valencia",
    "Centro de acopio en Barinas",
    "Andrea Lopez CI 22334455 desaparecida en Maracay",
    "Rescate urgente en Caracas edificio colapsado",
    "Centro de acopio en Mérida",
    "Miguel Torres CI 33445566 desaparecido en Valencia",
    "Rescate en Maracay urgente",
    "Centro de acopio en Caracas Chacao",
    "Juan Gomez CI 55667788 desaparecido en Barquisimeto",
    "Urgente rescate familia en Valencia",
    "Centro de acopio en Maracay",
    "Aid center in Caracas food water",
    "Rescue needed in Valencia urgent",
    "anaa lopezz desaparecida valencia",
    "ana lopez CI 12345678 desaparecida en Valencia otra vez"
]

import re
import uuid
from datetime import datetime

def extract_cedula(text):
    patrones = [
        r'ci\s*\d{6,8}',
        r'cedula\s*\d{6,8}',
        r'identidad\s*\d{6,8}',
        r'documento\s*\d{6,8}',
        r'\b\d{6,8}\b'
    ]

    for patron in patrones:
        match = re.search(patron, text.lower())
        if match:
            numero = re.search(r'\d{6,8}', match.group())
            if numero:
                return numero.group()

    return None

def extract_name(text):
    match = re.search(r'[A-Z][a-z]+ [A-Z][a-z]+', text)
    return match.group() if match else None

def categorize(text):
    text_lower = text.lower()

    desaparecido_keywords = [
        "desaparecid", "no aparece", "no lo conseguimos",
        "se busca", "extraviado", "perdido", "no sabemos de",
        "no responde", "desconocido paradero"
    ]

    rescate_keywords = [
        "rescate", "ayuda urgente", "auxilio", "atrapado",
        "necesitamos ayuda", "urgente", "emergencia",
        "no pueden salir", "quedaron dentro", "colapsado",
        "derrumb", "inund", "auxilien"
    ]

    acopio_keywords = [
        "centro de acopio", "donaciones", "recibiendo",
        "comida", "agua", "medicinas", "ropa",
        "llevar ayuda", "punto de ayuda", "recoleccion",
        "recoger insumos"
    ]

    for word in desaparecido_keywords:
        if word in text_lower:
            return "desaparecido"

    for word in rescate_keywords:
        if word in text_lower:
            return "necesita rescate"

    for word in acopio_keywords:
        if word in text_lower:
            return "centro de acopio"

    return "otro"

def extract_location(text):
    posibles = [
        # ciudades principales afectadas
        "Caracas", "La Guaira", "Valencia", "Maracay",
        "San Felipe", "Yumare", "Montalban", "Barquisimeto",
        "Caraballeda", "Catia La Mar",

        # estados (por si la gente escribe así)
        "Miranda", "Aragua", "Carabobo",
        "Falcon", "Falcón", "Yaracuy",
        "Lara", "Merida", "Mérida"
    ]

    for lugar in posibles:
        if lugar.lower() in text.lower():
            return lugar

    return None

In [80]:
def normalize_name(name):
    if not name:
        return None

    name = name.lower()

    replacements = {
        "zz": "z",
        "ss": "s",
        "ll": "l",
        "aa": "a",
        "ee": "e"
    }

    for k, v in replacements.items():
        name = name.replace(k, v)

    return name.title()


def generate_case_id(text, tipo, ubicacion):
    base = f"{tipo}_{ubicacion}_{text.lower()}"
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, base))

In [81]:
clean_data = []

for text in raw_data:
    cedula = extract_cedula(text)
    tipo = categorize(text)
    ubicacion = extract_location(text)

    record = {
        "id_caso": generate_case_id(text, tipo, ubicacion),
        "cedula": cedula,
        "tipo": tipo,
        "nombre": extract_name(text),
        "ubicacion": ubicacion,
        "fecha_registro": datetime.now().strftime("%Y-%m-%d")
    }

    clean_data.append(record)

In [82]:
import json

json_output = json.dumps(clean_data, indent=2, ensure_ascii=False)
print(json_output)

[
  {
    "id_caso": "62b98696-0a02-5f29-af16-82650aa0420f",
    "cedula": null,
    "tipo": "desaparecido",
    "nombre": null,
    "ubicacion": "Valencia",
    "fecha_registro": "2026-06-30"
  },
  {
    "id_caso": "f0660215-f7b4-5471-936e-2335dca4446e",
    "cedula": "23456789",
    "tipo": "desaparecido",
    "nombre": "Luis Martinez",
    "ubicacion": "Caracas",
    "fecha_registro": "2026-06-30"
  },
  {
    "id_caso": "ee26f336-9343-5424-9222-86ad3885f6d6",
    "cedula": null,
    "tipo": "necesita rescate",
    "nombre": null,
    "ubicacion": "Valencia",
    "fecha_registro": "2026-06-30"
  },
  {
    "id_caso": "38b58928-c97c-50ef-add0-6f86affab83e",
    "cedula": null,
    "tipo": "centro de acopio",
    "nombre": null,
    "ubicacion": "Barquisimeto",
    "fecha_registro": "2026-06-30"
  },
  {
    "id_caso": "d3ee0de6-b2a5-5035-aa1c-25e6402fbaaa",
    "cedula": "87654321",
    "tipo": "desaparecido",
    "nombre": "Maria Fernanda",
    "ubicacion": "Maracay",
    "fecha_re

In [83]:
deduplicated = {}

for item in clean_data:
    if item["cedula"]:
        key = item["cedula"]
    else:
        key = item["id_caso"]

    deduplicated[key] = item

clean_data = list(deduplicated.values())

print("Datos depurados:")
clean_data

Datos depurados:


[{'id_caso': '62b98696-0a02-5f29-af16-82650aa0420f',
  'cedula': None,
  'tipo': 'desaparecido',
  'nombre': None,
  'ubicacion': 'Valencia',
  'fecha_registro': '2026-06-30'},
 {'id_caso': 'f0660215-f7b4-5471-936e-2335dca4446e',
  'cedula': '23456789',
  'tipo': 'desaparecido',
  'nombre': 'Luis Martinez',
  'ubicacion': 'Caracas',
  'fecha_registro': '2026-06-30'},
 {'id_caso': 'ee26f336-9343-5424-9222-86ad3885f6d6',
  'cedula': None,
  'tipo': 'necesita rescate',
  'nombre': None,
  'ubicacion': 'Valencia',
  'fecha_registro': '2026-06-30'},
 {'id_caso': '38b58928-c97c-50ef-add0-6f86affab83e',
  'cedula': None,
  'tipo': 'centro de acopio',
  'nombre': None,
  'ubicacion': 'Barquisimeto',
  'fecha_registro': '2026-06-30'},
 {'id_caso': 'd3ee0de6-b2a5-5035-aa1c-25e6402fbaaa',
  'cedula': '87654321',
  'tipo': 'desaparecido',
  'nombre': 'Maria Fernanda',
  'ubicacion': 'Maracay',
  'fecha_registro': '2026-06-30'},
 {'id_caso': '8a2d1ebc-5ea9-5174-a285-aaf0310c16c8',
  'cedula': None,

In [84]:
with open("output.json", "w", encoding="utf-8") as f:
    f.write(json_output)

print("Archivo guardado")

Archivo guardado


In [85]:
import pandas as pd
from datetime import datetime

df = pd.DataFrame(clean_data)

fecha = datetime.now().strftime("%Y-%m-%d")
filename = f"output_{fecha}.xlsx"

df.to_excel(filename, index=False)

print(f"Archivo generado: {filename}")

Archivo generado: output_2026-06-30.xlsx


In [86]:
from google.colab import files
files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>